####1. Requirement
Read data from students_offline.csv file and load into offline_students_raw table.

In [0]:
offline_students_schema = "ID string, FirstName string, LastName string, Address string, Skills string, Contacts string"

offline_students_raw_df = (
    spark.read.format("csv")
    .option("header", True)
    .option("quote", "\"")
    .option("escape", "\"") 
    .schema(offline_students_schema)
    .load("/Volumes/dev/spark_db/datasets/spark_programming/data/students_offline.csv")
)

# since there are multiple , and " in the data, we need to escape them so that connector will understand the data correctly so that we can read data correctly

offline_students_raw_df.write.mode("overwrite").saveAsTable("dev.spark_db.offline_students_raw")

In [0]:
%sql
Select * FROM dev.spark_db.offline_students_raw

####2. Analysis Requirement
We want to know country wise student count.

In [0]:
%sql
  select id, from_json(address,
      """struct<AddressLine1 string,
        AddressLine2 string,
        City string,
        Country string,
        Pin string,
        State string>
      """) as address
  from dev.spark_db.offline_students_raw

In [0]:
%sql
/* Since address is in json format inside the table we'll first convert it to normal format by using from_json function in spark and provide the json schema to convert it(Here we convert it to struct trype object)*/
with offline_students(
  select id, from_json(address,
      """struct<AddressLine1 string,
        AddressLine2 string,
        City string,
        Country string,
        Pin string,
        State string>
      """) as address
  from dev.spark_db.offline_students_raw
)
select address.country, count(*) as count
from offline_students
group by address.country



####3. Requirement
Prepare an offline_students table which is ready for analysis.

We should not be loading complex data like json string inside the table like we did in the previous step because its not good practice and querying data using complex queries leads to performance issue so we should parse it once for and than create a table.

Complex Data Types in Spark
1. Struct - group related fields together
2. Array - ordered list of values
3. Map -key-value pairs

In [0]:
%sql
DROP TABLE IF EXISTS dev.spark_db.offline_students;

In [0]:
from pyspark.sql.functions import from_json

address_schema = "struct<AddressLine1 string, AddressLine2 string, City string, Country string, Pin string, State string>"
#[{"Skill":"Apache Spark","YearsOfExperience":"5"},{"Skill":"Apache Kafka","YearsOfExperience":"6"}] skill raw entry - This entry is array of 2 elements and each element is a struct ie array of struct.
skills_schema = "array<struct<Skill string, YearsOfExperience string>>"
# {"email":"xyz@abc.com","phone":"9823128923"} {"phone":"9873145698"} Contact field - Since there is no consistency in the data so its not struct. Its a map/dictionary or key value pair
contacts_schema = "map<string, string>"

offline_students_df = (
    offline_students_raw_df.withColumns({
        "address": from_json("address", address_schema),
        "skills": from_json("skills", skills_schema),
        "contacts": from_json("contacts", contacts_schema)
    })
)

#offline_students_df.display()
offline_students_df.write.mode("overwrite").saveAsTable("dev.spark_db.offline_students")


####4. Requirement
Perform the following analysis
1. What is country wise student count.
2. Find all students with more than 1 years of Spark knowledge
3. Find all students who didn't provide phone or whatsapp

4.1 What is country wise student count.

In [0]:
%sql
/* Can query the table without parsing to complex data type again and again*/
Select address.Country, count(*) as Count
FROM dev.spark_db.offline_students
Group By address.Country



4.2 Find all students with more than 1 years of Spark knowledge

In [0]:
%sql
select id, FirstName, LastName, explode(Skills) as skills
   from dev.spark_db.offline_students

In [0]:
%sql

with offline_students_skills(
   select id, FirstName, LastName, explode(skills) as skills
   from dev.spark_db.offline_students
)
select id, firstname, lastname, skills.*
from offline_students_skills
where skills.Skill like "%Spark%" and skills.YearsOfExperience > 1

4.3 Find all students who didn't provide phone or whatsapp

In [0]:
%sql
select ID, FirstName, LastName, contacts['email']
from dev.spark_db.offline_students
where contacts['phone'] is null and contacts['whatsapp'] is null